# MJO 0: Overview

Example diagrams from [Physical Science Lab](psl.noaa.gov/mjo)
 <p align="center">
    <img src="https://preview.weather.gov/images/arx/lanina/MJO2.jpg">
</p>

 <p align="center">
    <img src="https://psl.noaa.gov/mjo/psl/mjo/images/mjo_global_impacts.jpg">
</p>"

## MJO 1: Obtain Daily Model Data/Output

Read in daily model output via `daily.netcdf.ncl`"

`MJO_suite.py` - Line 44 - "OBTAINING DAILY OUTPUT"
```
generate_ncl_plots(os.environ["POD_HOME"]+"/daily_netcdf.ncl")
```

`daily_netcdf.ncl` reads daily output files from CAM2 and processes the data (each daily file has 30 days of data) (for example: U200(time, lat, lon))

Configuration file: `input_NCEP_Reanalysis_1980_1990.jsonc` (Configuration for MDTF-diagnostics driver script)

Additional details: [see here](https://mdtf-diagnostics.readthedocs.io/en/latest/sphinx/dev_cheatsheet.html#notes)

In [1]:
# Variable-specific environment variables which are accessed with os.environ()
# CASENAME =  ???? (string of file/case name)
# startdate = ???? (string in YYYYMMDD or YYYYMMDDHHMMSS for starting period of analysis)
# enddate =   ???? (string in YYYYMMDD or YYYYMMDDHHMMSS for ending period of analysis)

## DATADIR =  "data/CASENAME/day" (created in MJO_suite if doesn't already exist, L40)
# WORK_DIR =  (working directory) 

# U200_FILE = ???? (data file for 200-hPa zonal wind at lower boundary level, eastward wind in the atmosphere in m/s, scalar-coordinates where atmosphere pressure level lev=200)
# U850_FILE = ???? (data file for 850-hPa zonal wind at higher boundary level, eastward wind in the amotphere in m/s, scalar-coordinates where atmosphere pressure level lev=850)
# V200_FILE = ???? (data file for 200-hPa meridional wind, northward wind in the atmosphere in m/s, scalar-coordaintes where lev=200)
# V850_FILE = ???? (data file for northward wind in m/s, scalar-coordinates where atmospheric pressure level lev = 850)
# RLUT_FILE = ???? (data file for outgoing longwave radiation, aka: OLR)
# PR_FILE =   ???? (data file for percipitation)

# lev_coord =  "lev" (name of the level dimension, e.g. air pressure hPa (postive down, z-axis)
# lat_coord =  "lat" (name of the latitude dimension in the model's native format, e.g. latitude in degrees North, y-axis)
# lon_coord =  "lon" (name of the longitude dimension in the model's native format, e.g. longitude in degrees East, x-axis)
# time_coord = "time" (name of the time dimension in the model's native format)
# pr_var =     "PRECT" (name of precipitation rate, in m/s, with the dimensions = "time, "lat", "lon" -- retained for backwards compatibility -> absolute path to the file containing "pr" data, for example, "/dir/precip.nc")
# rlut_var =   "FLUT" (TOA Outgoing longwave flux/radiation, in W/m^2, with the dimensions = "time", "lat", "lon")

In [2]:
# # getenv == os.environ
import os

CASENAME = "QBOi.EXP1.AMIP.001"
startdate = '19790101'
enddate = '19811231'

# /glade/u/home/bundy/mdtf/MDTF_3_main/MDTF-diagnostics.blocking_notebook/diagnostics/MJO_suite/MJO_driver.py
#DATADIR = "/glade/u/home/bundy/diag/mdtf/inputdata/model/QBOi.EXP1.AMIP.001/"
WORK_DIR = os.getcwd()
DATADIR = "/data/"

U200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U200.day.nc"
V200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V200.day.nc"
U850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U850.day.nc"
V850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V850.day.nc"
RLUT_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.FLUT.day.nc"
PR_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.PRECT.day.nc"

lev_coord = "lev"
lat_coord = "lat"
lon_coord = "lon"
time_coord = "time"
pr_var = "PRECT"
rlut_var = "FLUT"

u200_var = "U200"
v200_var = "V200"

wk_dir = WORK_DIR + "/model/"

# setup data directory, if does not already exist
#if not os.path.exists(DATADIR): os.makedirs(DATADIR)

In [3]:
# check date format
from datetime import datetime

def check_date_format(date_in="", is_start=True):
    # update and check date format

    # if date string is empty
    if date_in == "":
        print(f"ERROR: No date_type given to function check_date_format()")
        return

    # if input is just YYYY, add in MMDDHH suffix
    if len(date_in) == 4:
        # if input if just YYYY, add MMDDHH
        if is_start: # is start date
            date_suffix = str(101 * 100) # 10 10 0 # TODO: month as 1 or 01?
        if not is_start: # is end date
            date_suffix = str(1231 * 100) # 12 31 00

        date_out = date_in + date_suffix
        #print(f"Corrected {date_in} date date to {date_out}")

    # if input is YYYYMMDD, but missing hours
    if len(date_in) == 8:
        date_out = date_in + "00"
        #print(f"Corrected {date_in} date to {date_out}")

    # if input is correct length, check format
    if len(date_in) == 10:
        date_out = date_in

    date_format = "%Y%m%d%H" # YYYYMMDDHH
    try:
        matches = bool(datetime.strptime(date_out, date_format))
    except ValueError:
        matches = False
    #print(f"{date_in} matches {date_format} = {matches}")
    #print(f"Final Date = {date_out}")
    return date_out

startdate = check_date_format(startdate, is_start=True)
enddate = check_date_format(enddate, is_start=False)

In [4]:
# daily_netcdf.ncl
# netCDF variables

print("Starting: Daily NETCDF")
casename = CASENAME
datadir = DATADIR
level = lev_coord
wk_dir = WORK_DIR + "/model/"
if not os.path.exists(wk_dir): os.makedirs(wk_dir)

file_u200 = U200_FILE
file_v200 = V200_FILE
file_u850 = U850_FILE
file_v850 = V850_FILE
file_rlut = RLUT_FILE
file_pr = PR_FILE

print(f"daily_netcdf.ncl reading {file_pr} for time coordinates")
print("Assuming without checking that all have time coordinates")

Starting: Daily NETCDF
daily_netcdf.ncl reading /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc for time coordinates
Assuming without checking that all have time coordinates


In [5]:
import xarray as xr
file_netcdf = xr.open_dataset(file_pr)
print(list(file_netcdf.coords))
file_netcdf

['time', 'lat', 'lon']


<xarray.Dataset> Size: 565MB
Dimensions:    (time: 2555, nbnd: 2, lat: 192, lon: 288)
Coordinates:
  * time       (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    PRECT      (time, lat, lon) float32 565MB ...

In [6]:
print(file_netcdf.variables.keys())
#print("\n")
#print(file_netcdf.variables)

KeysView(Frozen({'date': <xarray.Variable (time: 2555)> Size: 10kB
[2555 values with dtype=int32]
Attributes:
    long_name:  current date (YYYYMMDD), 'time_bnds': <xarray.Variable (time: 2555, nbnd: 2)> Size: 41kB
[5110 values with dtype=object]
Attributes:
    long_name:  time interval endpoints, 'PRECT': <xarray.Variable (time: 2555, lat: 192, lon: 288)> Size: 565MB
[141281280 values with dtype=float32]
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:              m/s
    long_name:          Total (convective and large-scale) precipitation rate...
    cell_methods:       time: mean                                           ..., 'time': <xarray.IndexVariable 'time' (time: 2555)> Size: 20kB
array([cftime.DatetimeNoLeap(1975, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 3, 0, 0, 0, 0, has_year_zero=True), ...,
       cftime.DatetimeNoLeap(1981, 12, 29, 0, 0, 0, 0, has_year_z

In [7]:
yr1 = int(startdate)
yr2 = int(enddate)
print(yr1)
print(yr2)

1979010100
1981123100


In [8]:
time = file_netcdf.variables[time_coord]
print(time.attrs)
# TODO: find units/calendar, not visible with xarray, but below uses netCDF4
time

{'long_name': 'time', 'bounds': 'time_bnds'}


<xarray.IndexVariable 'time' (time: 2555)> Size: 20kB
array([cftime.DatetimeNoLeap(1975, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 3, 0, 0, 0, 0, has_year_zero=True), ...,
       cftime.DatetimeNoLeap(1981, 12, 29, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 30, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 31, 0, 0, 0, 0, has_year_zero=True)],
      shape=(2555,), dtype=object)
Attributes:
    long_name:  time
    bounds:     time_bnds

In [9]:
from netCDF4 import Dataset
file_pr_as_dataset = Dataset(file_pr)
time_netcdf = file_pr_as_dataset[time_coord]

In [10]:
print(f"before: \n{time_netcdf.units}\n{time_netcdf.calendar}")

if (time_netcdf.units == "julian day"): # if time units in Julian days, convert
    time_netcdf.units = "days since -4713-01-01 00:00:00"
    time_netcdf.calendar = "julian"

print(f"after: \n{time_netcdf.units}\n{time_netcdf.calendar}")

before: 
days since 1975-01-01 00:00:00
noleap
after: 
days since 1975-01-01 00:00:00
noleap


In [11]:
import numpy as np
days_since_date = time[:] # extract all time days since date
days_since_date

<xarray.IndexVariable 'time' (time: 2555)> Size: 20kB
array([cftime.DatetimeNoLeap(1975, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1975, 1, 3, 0, 0, 0, 0, has_year_zero=True), ...,
       cftime.DatetimeNoLeap(1981, 12, 29, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 30, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(1981, 12, 31, 0, 0, 0, 0, has_year_zero=True)],
      shape=(2555,), dtype=object)
Attributes:
    long_name:  time
    bounds:     time_bnds

In [12]:
# select data for a single year
# file_netcdf.sel(time="1980")

In [13]:
# convert mixed Julian/Gregorian date to a UT-referenced date
# via NCL: cd_calendar --> cd_calendar(time, -3)
# Returns values in YYYYMMDDHH as an integer

# from datetime import datetime, timedelta

# start date as datetime
#start_datetime = datetime(1975, 1, 1, 0, 0, 0)

# collect all datetimes
#time_all = []
#for day_since_start in days_since_date:
#    next_day = (start_datetime + timedelta(days=day_since_start))
#    day_YYYYMMDDHH = next_day.strftime("%Y%m%d%H")
#    time_all.append(int(day_YYYYMMDDHH))
#print(f"Start date = {time_all[0]}, End date = {time_all[-1]}, number of dates = {len(time_all)}")

In [14]:
i1 = 0
i2 = len(time) - 1
print(f"From {i1} to {i2}")
print(f"Time range in file: {time[i1].values} - {time[i2].values}")

From 0 to 2554
Time range in file: 1975-01-01 00:00:00 - 1981-12-31 00:00:00


In [15]:
# confirm date format
start_time = check_date_format(str(yr1), is_start=True)
end_time = check_date_format(str(yr2), is_start=False)

In [16]:
# 24 hours tolerance allows for different time resolutions

#tol = 24

#for i in range(len(time_all)-1):
#    #print(f"examining times {i} {time_all[i]}")
#    if abs(int(time_all[i]) - int(start_time)) < tol:
#        i1 = i
#        print(f"Found start_time {time_all[i]} {end_time}")
#    if abs(int(time_all[i]) - int(end_time)) < tol:    
#        i2 = i
#        print(f"Found end_time {time_all[i]} {end_time}")

#print(f"Time range indices: {i1} {time_all[i1]} - {i2} {time_all[i2]}")

In [17]:
filtered_file = time[i1:i2]
print(f"Using date range: {min(filtered_file).values} to {max(filtered_file).values}")
ndays = len(filtered_file)
print(f"Number of days: {ndays}")

Using date range: 1975-01-01 00:00:00 to 1981-12-30 00:00:00
Number of days: 2554


In [18]:
# check if data array is monotonic (strictly increasing)
import pandas as pd
def check_monotonic(array):
    # Check for increasing order sequence
    is_increasing = pd.Index(array).is_monotonic_increasing
    try:
        assert is_increasing
        return is_increasing
    except Exception:
        print("ERROR: daily_netcdf.ncl finds dates are not monotonic increasing")
    
print(f"Is monotonic = {check_monotonic(filtered_file)}")

Is monotonic = True


In [19]:
# First just use read_model_file for pr here, then put into var loop below
from netCDF4 import Dataset

def read_dim(f, var_name, opt=None, upper_bound=None, lower_bound=None):
    print("running read_dim")
    # filter lat bounds from 40 to -40
    routine_name = "read_dim"
    verbose = False
    if opt:
        if lower_bound is not None:
            #print(f"found lower_bound {lower_bound}")
            pass
        if upper_bound is not None:
            #print(f"found upper_bound {upper_bound}")
            pass
    
    if lower_bound is not None and upper_bound is not None:
        # filter based on upper and lower bound
        var = f[var_name].where((f[var_name] > lower_bound) & (f[var_name] < upper_bound))
    else:
        var = f[var_name]

    return var    

def get_gw(f, latS, latN):
    # gaussian weight for the latitude dimension
    if False:
    #if "gw" in file_netcdf.coords:
        # remove any existing "gw" dimension
        #f.drops_vars("gw")
        #new_gw = f[gw[lat_coord|latS:latN]]
        print("TODO")
    else:
        print("running get_gw")
        lat = read_dim(f, lat_coord, opt=False, upper_bound=latN, lower_bound=latS) # cut full range based on upper and lower bounds
        nlat = lat.count().item()
        slat = xr.DataArray(coords=(range(nlat+1), ))
        #gw = xr.DataArray(coords=(range(nlat+1), ), dims="gw")
        new_gw = {}

        if lat[0].values < lat[1].values:
            slat[0] = -90.0
            slat[nlat] = 90.0
        else:
            slat[0] = 90.0
            slat[nlat] = -90.0

        for i in range(nlat-1):
            slat[i] = (lat[i-1].values + lat[i].values)/2

        for i in range(nlat-1):
            new_gw[i] = abs(np.sin(slat[i+1] / (180*np.pi)) - np.sin(slat[i] / (180 * np.pi)))
        
    return new_gw

def pinterp(var_name=None, plev=None, fin=None, fin_ps=None):
    print(f"pinterop() interpolating {var_name} to pressure level {plev} mbar")        

def read_model_file(var_name_in=None, file_in=None,
                   var_name_out=None, file_out=None,
                   delete_existing=False,
                   i1=None, i2=None, time_coord=None,
                   lat_coord=None, lon_coord=None,
                   date=None, interp_opts=None,
                   var_name_in_3d=None, plev=None, file_3d=None, file_ps=None):
    # utils.ncl - L186
    # re-write model file
    # writing a file with only necessary time dimension and lat bounds and adding in gw
    # interp_opts = optional args for interpolating in space
    # var_name_in_3d, plev, file_3d, file_ps = optional args for pressure interp
    print("running read_model_file")

    latS = -40
    latN = 40
    opt = True
    lower_bound = latS
    upper_bound = latN

    if os.path.exists(file_out) and not delete_existing: # do not write new file
        # TODO: FIX
        print("file output exists and delete_existing is False")
        print(f"WARNING: using existing file {file_out}")
        print(f"To override, change delete_existing to True")
        if interp_opts is not None:
            #f_out_for_interp = Dataset(file_out)
            f_out_for_interp = xr.open_dataset(file_out)
            var = f_out_for_interp[var_name_out]
            gw = get_gw(f_out_for_interp, latS, latN)
            interp_save_res(var_name_out, interp_opts, var["lat"], var["lon"], gw)
            del(f_out_for_interp)
    else:
        print("write new file")
        if os.path.exists(file_out):
            print(f"WARNING: over-writing existing file, removing existing {file_out}")
            os.remove(file_out)
        if os.path.exists(file_in):
            # CURRENTLY RUNNING:
            print(f"opening {var_name_in} from {file_in}")
            f_in = xr.open_dataset(file_in)
            lat = read_dim(f_in, lat_coord, opt, upper_bound=upper_bound, lower_bound=lower_bound)
            opt = False
            lon = read_dim(f_in, lon_coord, opt, upper_bound=upper_bound, lower_bound=lower_bound)
            gw = get_gw(f_in, latS, latN)

            print("FILE INPUT (while writing new file):")
            print(var_name_in)
            time_start = f_in[var_name_in][i1].time.values
            time_end = f_in[var_name_in][i2].time.values
            print(time_start, time_end)
            ds = f_in[var_name_in]
            #ds = ds.expand_dims({"gw": gw}) -> makes file too large (21 GB)
            var = ds.sel(time=slice(time_start, time_end), lat=slice(-40, 40)) # L245 (utils.ncl)
            print(var)
        else:
            print(f"WARNING: file does not exist for MJO diagnostics: {file_in}")
            print(f"Looking for file matching {var_name_in_3d} at plev {plev}")
            
            # needs checks for var_name_in_3d, plev and file_ps
            if file_ps is None:
                print(f"ERROR: can't find required PS file: {file_ps}")
                return

            f_3d = xr.open_dataset(file_3d)
            f_ps = xr.open_dataset(file_ps)
            # TODO: fill out pinterp() function
            var_1 = pinterp(var_namae_in_3d, plev, f_3d, f_ps) #TODO
            var = ds.sel(time=slice(time_start, time_end), lat=slice(-40, 40)) # L245 (utils.ncl)
            
            if interp_opts:
                print("INTERP_OPTS IS TRUE")
                # TODO

    # Writing new file
    print(f"Creating output file {file_out}")
    try:
        file_out.close()
    except:
        pass
    
    print(f"WRITING OUT NEW NETCDF FILE: {file_out}")
    file_output_xarray = var[time_coord]
    file_output_xarray = file_output_xarray.expand_dims({"gw": gw}) # temp solution: write to a new file
    print(f"file_output_xarray = {file_output_xarray}")
    file_output_xarray.to_netcdf(file_out)

In [20]:
# precipitation rate (pr)
print(f"daily_netcdf.ncl reading {file_pr} for making precip file!\n")
delete_existing = True # True = overwrite, False = don't overwrite

if os.path.exists(file_pr):
    print(f"found pr input file = {file_pr}")
    var_name = pr_var
    var_name_out = "pr"
    file_in = file_pr
    file_out = wk_dir + CASENAME + "." + var_name_out + ".day.nc"
    print(f"file_out = {file_out}\n")
    var_name_3d_model = "not provided"
    file_in_3d = "not provided"
    file_in_ps = "not provided"
    plev = 0

    interp_opts = True # store this field as base resolution no matter what
    interp_to_var_name = var_name_out
    read_model_file(var_name, file_in, var_name_out, file_out, 
                    delete_existing, i1, i2, 
                    time_coord, lat_coord, lon_coord, filtered_file, interp_opts,
                    var_name_3d_model, plev, file_in_3d, file_in_ps)
else:
    print("ERROR: daily pr input files does not exist for MJO diagnostics")

daily_netcdf.ncl reading /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc for making precip file!

found pr input file = /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc
file_out = /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.pr.day.nc

running read_model_file
write new file
opening PRECT from /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.PRECT.day.nc
running read_dim
running read_dim
running get_gw
running read_dim
FILE INPUT (while writing new file):
PRECT
1975-01-01 00:00:00 1981-12-31 00:00:00
<xarray.DataArray 'PRECT' (time: 2555, lat: 84, lon: 288)> Size: 247MB
[61810560 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 672B -39.11 -38.17 -37.23 ... 37.23 38.17 39.11
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:           

In [21]:
plevs = [850, 200]
var_names = ["u", "v"]

file_u200 = U200_FILE
file_v200 = V200_FILE
file_u850 = U850_FILE
file_rlut = RLUT_FILE
file_pr = PR_FILE

for i in range(len(plevs)):
    plev = plevs[i]
    for j in range(len(var_names)):
        var_name = var_names[j].upper() + str(plev)
        var_name_3d_model = var_names[j] + "_var" # as read from history files (3d field)
        var_name_plev_model = var_names[j] + str(plev) + "_var" # as read from history files (pressure slice)
        var_name_plev_package = var_names[j] + str(plev) # new file name and varname in file
        file_var_name = var_names[j].upper() + str(plev) + "_FILE"
        if file_var_name == "U850_FILE":
            file_in = file_u850
        if file_var_name == "V850_FILE":
            file_in = file_v850
        if file_var_name == "U200_FILE":
            file_in = file_u200
        if file_var_name == "V200_FILE":
            file_in = file_v200
        print(f"file_in = {file_in}")
        file_out = wk_dir + CASENAME + "." + var_name_plev_model + ".day.nc"
        print(f"\tfile_out   = {file_out}")
        # all files supplied to POD are 3D slices -> PS not part of varlist request
        file_in_3d = WORK_DIR + DATADIR + CASENAME + "." + var_name_3d_model + ".day.nc"
        #print(f"\tfile_in_3d = {file_in_3d}")
        file_in_ps = WORK_DIR + DATADIR + CASENAME + ".PS.day.nc"
        #print(f"\tfile_in_ps = {file_in_ps}")
        
        read_model_file(var_name, file_in, var_name_out, file_out, 
                    delete_existing, i1, i2, 
                    time_coord, lat_coord, lon_coord, filtered_file, interp_opts,
                    var_name_3d_model, plev, file_in_3d, file_in_ps)

file_in = /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.U850.day.nc
	file_out   = /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.u850_var.day.nc
running read_model_file
write new file
opening U850 from /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.U850.day.nc
running read_dim
running read_dim
running get_gw
running read_dim
FILE INPUT (while writing new file):
U850
1975-01-01 00:00:00 1981-12-31 00:00:00
<xarray.DataArray 'U850' (time: 2555, lat: 84, lon: 288)> Size: 247MB
[61810560 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 672B -39.11 -38.17 -37.23 ... 37.23 38.17 39.11
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Attributes:
    Sampling_Sequence:  rad_lwsw
    units:              m/s
    long_name:          Zonal wind at 850 mbar pressure surface
    cell_methods:       time: mean                           

In [22]:
# outgoing longwave radiation (rlut)

if os.path.exists(file_rlut):
    var_name_out = "rlut"
    var_name = rlut_var
    file_in = file_rlut
    print(f"file_in = {file_in}")
    file_out = wk_dir + CASENAME + "." + var_name_out + ".day.nc"
    print(f"\tfile_out   = {file_out}")
    var_name_3d_model = "not provided"
    file_in_3d = "not provided"
    file_in_ps = "not provided"
    plev = 0

    read_model_file(var_name, file_in, var_name_out, file_out, 
                    delete_existing, i1, i2, 
                    time_coord, lat_coord, lon_coord, filtered_file, interp_opts,
                    var_name_3d_model, plev, file_in_3d, file_in_ps)
else:
    print("daily rlut file does not exist for MJO diagnostics")

file_in = /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.FLUT.day.nc
	file_out   = /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.rlut.day.nc
running read_model_file
write new file
opening FLUT from /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.FLUT.day.nc
running read_dim
running read_dim
running get_gw
running read_dim
FILE INPUT (while writing new file):
FLUT
1975-01-01 00:00:00 1981-12-31 00:00:00
<xarray.DataArray 'FLUT' (time: 2555, lat: 84, lon: 288)> Size: 247MB
[61810560 values with dtype=float32]
Coordinates:
  * time     (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 672B -39.11 -38.17 -37.23 ... 37.23 38.17 39.11
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Attributes:
    cell_methods:       time: mean
    long_name:          Upwelling longwave flux at top of model
    units:              W/m2
    Sampling_Sequence:  rad_lwsw
Creating output file /home/cs

# MJO 2: Compute the Daily Anomalies

Via `daily_anom.ncl`

In [23]:
routine_name = "daily_anom"
yr1 = int(startdate)
yr2 = int(enddate)

# encode datetime with milliseconds
start_date = (yr1 * 10000)+201       # 19790101 -> 197901010201
end_date = ((yr2 + 1) * 10000) + 101 # 19811231 -> 198112320101
print(f"Date from {start_date} to {end_date}")

delete_existing = True # True = overwrite, False = don't overwrite
if delete_existing: 
    anom_files = [file for file in os.listdir(wk_dir) if ".anom" in file]
    if len(anom_files) != 0:
        print("\nWARNING: daily_anom.ncl deleting existing anom files\n")
    var_files = [file for file in os.listdir(wk_dir) if ".nc" in file]
    print(f"({wk_dir}) contains 'anom': {anom_files}")   
    #print(f"({wk_dir}) contains '.nc' : {var_files}")   

Date from 19790101000201 to 19811231010101
(/home/cs/Github/geocat-research/mjo/model/) contains 'anom': []


In [24]:
file_netcdf = xr.open_dataset(wk_dir + "QBOi.EXP1.AMIP.001.pr.day.nc")
file_netcdf

<xarray.Dataset> Size: 2MB
Dimensions:                        (gw: 83, time: 2555)
Coordinates:
  * gw                             (gw) int64 664B 0 1 2 3 4 ... 78 79 80 81 82
  * time                           (time) object 20kB 1975-01-01 00:00:00 ......
Data variables:
    __xarray_dataarray_variable__  (gw, time) object 2MB ...
Attributes:
    __xarray_dataarray_name__:  time

In [25]:
# convert mixed Julian/Gregorian date to a UT-referenced date
# via NCL: cd_calendar --> cd_calendar(time, -2)
# Returns values in YYYYMMDD as an integer
#def cd_calendar(date):
#    from datetime import datetime, timedelta

    # start date as datetime
#    start_datetime = datetime(1975, 1, 1, 0, 0, 0)
    
    # collect all datetimes
#    time_all = []
#    for day_since_start in days_since_date:
#        next_day = (start_datetime + timedelta(days=day_since_start))
#        day_YYYYMMDD = next_day.strftime("%Y%m%d")
#        time_all.append(int(day_YYYYMMDD))
#    
#    print(f"Start date = {time_all[0]}, End date = {time_all[-1]}")
#    return time_all

In [26]:
var_names = ["pr", "rlut", "u200", "u850", "v200", "v850"]

for i in range(len(var_names)):
    var_name = var_names[i]
    print(f"{routine_name} starting variable {var_name}")
    file_in = wk_dir + CASENAME + "." + var_name + ".day.nc"
    print(f"\tFile in:  {file_in}")
    file_out = wk_dir + CASENAME + "." + var_name + ".day.anom.nc"
    print(f"\tFile out: {file_out}")

    if os.path.exists(file_out) and not delete_existing:
        print(f"{file_out} already exists. To force recalculation, set daily_anom.ncl delete_existing = True\n")
    else:
        if os.path.exists(file_out) and delete_existing:
            print(f"{file_out} already exists. Overwriting since delete_existing = True\n")
            os.remove(file_out)

        if os.path.exists(file_in):
            print(f"reading {file_in}\n")
            input_file = xr.open_dataset(file_in)
            print(input_file)
            time = input_file[time_coord]
            gw = input_file["gw"]
            break
            #date = cd_calendar(time)''' ## YYYYMMDD

daily_anom starting variable pr
	File in:  /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.pr.day.nc
	File out: /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.pr.day.anom.nc
reading /home/cs/Github/geocat-research/mjo/model/QBOi.EXP1.AMIP.001.pr.day.nc

<xarray.Dataset> Size: 2MB
Dimensions:                        (gw: 83, time: 2555)
Coordinates:
  * gw                             (gw) int64 664B 0 1 2 3 4 ... 78 79 80 81 82
  * time                           (time) object 20kB 1975-01-01 00:00:00 ......
Data variables:
    __xarray_dataarray_variable__  (gw, time) object 2MB ...
Attributes:
    __xarray_dataarray_name__:  time


In [27]:
input_files = [file for file in os.listdir(WORK_DIR + DATADIR) if ".nc" in file]
input_files

['QBOi.EXP1.AMIP.001.V850.day.nc',
 'QBOi.EXP1.AMIP.001.U850.day.nc',
 'QBOi.EXP1.AMIP.001.U200.day.nc',
 'QBOi.EXP1.AMIP.001.V200.day.nc',
 'QBOi.EXP1.AMIP.001.FLUT.day.nc',
 'QBOi.EXP1.AMIP.001.PRECT.day.nc']

In [30]:
from geocat.comp import climate_anomaly

delete_existing = True

# set up anomaly directory
anom_dir = WORK_DIR + DATADIR + "anomaly/"
if not os.path.exists(anom_dir): os.makedirs(anom_dir)

for input_nc in input_files:
    file_nc = input_nc.split(".")[4]
    var_name = file_nc.lower()
    file_in = WORK_DIR + DATADIR + input_nc
    file_out = anom_dir + CASENAME + "." + var_name + ".day.anom.nc"
    print(f"Calculating anomaly for {file_in}")

    # calculate daily anomaly
    if delete_existing:
        input_xarray = xr.open_dataset(file_in)
        
        # remove encoding which prevents anom file from being saved
        #if "time" in input_xarray.coords: 
        #    input_xarray["time"].encoding["units"] = ""
        #if "time_bnds" in input_xarray.coords:
        #    input_xarray["time_bnds"].encoding["units"] = ""
        

        if "time_bnds" in input_xarray.coords:
            #input_xarray = input_xarray.drop_vars("time_bnds")
            del input_xarray["time_bnds"].encoding

        print(list(input_xarray.data_vars))
        
        climate_anom_xarray = climate_anomaly(input_xarray, freq="day")
        print(f"\tSaving anomaly file {file_out}")
        climate_anom_xarray.to_netcdf(file_out)
    else:
        print("Keeping existing anom files")

Calculating anomaly for /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.V850.day.nc
['date', 'V850']
	Saving anomaly file /home/cs/Github/geocat-research/mjo/data/anomaly/QBOi.EXP1.AMIP.001.v850.day.anom.nc
Calculating anomaly for /home/cs/Github/geocat-research/mjo/data/QBOi.EXP1.AMIP.001.U850.day.nc
['date', 'time_bnds', 'U850']
	Saving anomaly file /home/cs/Github/geocat-research/mjo/data/anomaly/QBOi.EXP1.AMIP.001.u850.day.anom.nc


KeyError: 'days since 1975-01-01 00:00:00s'

In [ ]:
# "compute_daily_anom() - L61
# function in calc_utils.ncl - L371

#import xarray as xr
#from geocat.comp import climate_anomaly
#from matplotlib import pyplot as plt
#import datetime

#def compute_daily_anom(data_as_nc, is_plot=False):
    # compute the daily anomaly
    # See; calcDayAnom in geocat-applications
#    ds = xr.open_dataset(data_as_nc)
    # calculates climate anomalies by subtracting the long-term mean of each 'freq' period (e.g. 'day')
    #calcDayAnomTLL = climate_anomaly(dset=ds.aice_d,
      #                               freq="day",
     #                               tim_dim=None) # anomalies from daily climatology

    # optional plotting
    #if is_plot:
    #    calcDayAnomTLL[0, ;, ;].assign_attrs(long_name="CAM2 Anomaly").plot()
    #return calcDayAnomTLL

#### Plot Anomaly

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_anomaly(ds, vars_type):
    var_average = ds.sel(time="1980")[var_type.upper()].mean(dim="time")
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    var_average.plot.contourf(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', cbar_kwargs={'label': 'Average Value'})

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)

    plt.title(f"Average Anomaly {var_type.upper()} in 1980")

    plt.show()

In [ ]:
file_dir = WORK_DIR + DATADIR + "anomaly/"
anom_files = [file for file in os.listdir(file_dir) if "anom" in file]
for anom in anom_files:
    data_xarray = xr.open_dataset(file_dir + anom)
    var_type = anom.split(".")[4]
    plot_anomaly(data_xarray, var_type)

# MJO 3: Calculate EOFs

Via `mjo_EOF.ncl`

Calculating the empirical orthognal functions (EOFs)

- [eofunc_eofs](https://geocat-comp.readthedocs.io/en/v2023.02.0/user_api/generated/geocat.comp.stats.eofunc_eofs.html): computes the empirical orthogonal functions (EOFs, aka: Principal Component Analysis. Python alterative to NCL function `eofunc`, built on [`eofs`](https://ajdawson.github.io/eofs/) package
- [geocat-example: NCL_eof_1_1.py](https://geocat-examples.readthedocs.io/en/latest/gallery/Contours/NCL_eof_1_1.html)

In [ ]:
# input data
file_dir = WORK_DIR + DATADIR + "anomaly/"
anom_files = [file for file in os.listdir(file_dir) if "anom" in file]
anom_files

In [ ]:
# for example:
xr.open_dataset(file_dir + anom_files[0])

In [ ]:
# EOF variables
seasons = ["winter", "summer"]

neof = 4
latS = -30
latN = 30

pltDir = file_dir + "PS/" # directory for plots
pltType = "ps"

if not os.path.exists(pltDir): os.makedirs(pltDir)
print(pltDir)

In [ ]:
# calculate Guassian weights
#TODO

### Fixed Lanczos Filter Weights

Create bandpass filter from weights

See: [Lanczos Filter Weights](https://www.ncl.ucar.edu/Applications/mjoclivar.shtml)

```
When needed, the weights for the suggested 20-100 day bandpass Lanczos filter are generated 'on-the-fly' using:

  ihp      = 2                             ; bpf=>band pass filter
  nWgt     = 201
  sigma    = 1.0                           ; Lanczos sigma
  fca      = 1./100.
  fcb      = 1./20.
  wgt      = filwgts_lanczos (nWgt, ihp, fca, fcb, sigma )

```

See `generate_lanczos_filter_weights.ncl` which generates `lanczos_filter_weights_output.txt`

In [ ]:
def lanczos_filter_weights(nwt=None, ihp=None, fca=None, fcb=None, nsigma=None):
    # nwt = scale indicating the total number of weights (must be an odd number, nwt >= 3).The more weights, the better the filter, but greater loss of data
    # ihp = scale indicating the low-pass filter
    # fca = scale indicating the cut-off frequency of the ideal high or low-pass filter (0 < fca < 0.5)
    # fcb = scale used only when band-pass filter is desired, second cut-off frequency (fca < fcb < 0.5)
    # scale indicating the power of sigma factor (nsigma >= 0), ngima = 1 is common
    # returns: a symmetrical set of weights
    ncl_output_weights = np.loadtxt("lanczos_filter_weights_output.txt", delimiter=",", skiprows=17)
    return ncl_output_weights 

In [ ]:
## calculate one-dimensional filter weights (filwgts_lanczos) 

# create BandPass filter
#ihp = 2 # bpf->band pass filter, 2 = band-pass
#nWgt = 201 # number of weights (must be an odd number, the more weights, the better the filter, but the greater loss of data)
#sigma = 1 # lanczos sigma
#fca = 1/100 # indicating the cut-off frequency of the ideal high/low-pass filter (0.0 < fca < 0.5)
#fcb = 1/20 # scalar used when band-pass filter is desired. It is the second cut-off frequency  (fca < fcb < 0.5)
weights = lanczos_filter_weights()
weights

In [ ]:
for anom in anom_files:
    # read anomalies
    ivars = anom.split(".")[4].upper()
    print(f"Anomaly File: {file_dir + anom}")
    print(f"Starting: {ivars}")
    anom_netcdf = xr.open_dataset(file_dir + anom)

    time = anom_netcdf[ivars]
    '''
    # collect only the year
    date_start = datetime.strptime(str(time["time"].min().item()), "%Y-%m-%d %H:%M:%S")
    date_end = datetime.strptime(str(time["time"].max().item()), "%Y-%m-%d %H:%M:%S")
    # L102 (mjo_EOF.ncl) -> get year from datetime string
    yrStrt = date_start.year
    yrLast = date_end.year
    print(yrStrt)
    print(yrLast)
    print("\n")
    '''
    yrStrt = time["time"].min().item()
    yrLast = time["time"].max().item()

    X = anom_netcdf.sel(time=slice(yrStrt, yrLast), lat=slice(latS, latN)) # L96 (mjo_EOF.ncl)
    lon = X["lon"] # L107 (mjo_eof.ncl)
    lat = X["lat"] # L108 (mjo_eof.ncl)
    # gw = gw.sel(lat=slice(latS, latN)) # TODO
    mlon = len(lon)
    nlat = len(lat)

    # apply the band pass filter to the original anomalies
    # L121 (mjo_EOF.ncl)
    # wgt_runave_wrap: calculates a weighted running average on the rightmost dimension and retains metadata
    rightmost_dim = list(X.sizes.keys())[-1]
    #print(f"rightmost dimension (fastest varying dimension): {rightmost_dim}\n")
    #print(X[rightmost_dim])
    #rolling_window = X[var_type].rolling(time=len(weights), center=True) # define the rolling window over the 'lon' dimension (centering the window)
    #weighted_rolling_window = rolling_window.weighted(xr.DataArray(weights, dims=["rolling_window"]))
    #weighted_running_average = weighted_rolling_window.mean("rolling_window") # retains original metadata
    print("Calculating weighted running average...")
    weighted_running_average = X[ivars].rolling(time=len(weights), center=True).mean().dropna("time")
    x = weighted_running_average
    #print(weighted_running_average)

    # remove means of band pass series (NOT NECESSARY)
    # X = dim_rmvmean(x) # TODO L126 (mjo_EOF.ncl)

    # Two seasons, November to April and May to October
    count = [0, 0]
    for t in time:
        date_time = t["time"].values
        month = datetime.strptime(str(date_time), "%Y-%m-%d %H:%M:%S").month
        if month >= 5 and month <= 10:
            count[1] = count[0] + 1
        else:
            count[0] = count[0] + 1

    time0 = np.empty(count[0]+1, dtype="S20")
    time1 = np.empty(count[1]+1, dtype="S20")

    if len(time) > 1000:
        time_warning = f"{len(time)} time samples, may take a while, Check {pltDir}"
    else:
        time_warning = f"no time warning, nt {len(time)}"
    
    for i, season in enumerate(seasons):
        print(f"Computing {ivars}, {season}: {time_warning}")
        pltName = f"{CASENAME}.MJO.EOF.{ivars}.{season}"
        print(pltName)

        #x_season = np.zeros((count[i], nlat, mlon))
        #x_season = xr.DataArray()

        if i == 0:
            index = 0
            # set x_season to dataarray from April to November (Summer)
            #print(x.time)
            # TODO: fix range: less than 4 and greater than 11
            x_season = x.sel(time=x.time.dt.month.isin(range(4, 12)))
        else:
            pass
            '''
            index = 0
            for n, t in enumerate(time):
                date_time = t["time"].values
                month = datetime.strptime(str(date_time), "%Y-%m-%d %H:%M:%S").month
                if month >= 5 and month <= 10:
                    time1[index] = str(datetime.strptime(str(date_time), "%Y-%m-%d %H:%M:%S"))
                    x_season[index,:,:] = x.isel(time=n) # L176 (mjo_EOF.ncl)
                    index += 1
            '''
        #x_season["time"] = time1

        # compute EOFs: no need to areal weight (15S to 15N)
        break

    print("\n")

# MJO 3.5: Phase Diagrams Base (WIP)

Phase diagram split into 8 equal parts for example:

 <p align="center">
    <img src="https://psl.noaa.gov/mjo/mjoindex/romi_cpcolr_phase_diag.png">
</p>

In [ ]:
# Create base of the map for a Phase Diagram
from matplotlib import pyplot as plt
import matplotlib.lines as lines # add phase division lines
import numpy as np

def mjo_basemap(is_darkmode=False, start_date="Start", end_date="End"):
    if is_darkmode:
        plt.style.use("dark_background") # dark mode for testing
        color_dm = "white"
    else:
        plt.style.use("default")
        color_dm = "black"
    
    fig = plt.figure(figsize=(15, 15))
    ax = fig.add_subplot(111)
    
    # set x and y axis labels
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax2 = ax.twinx() 
    ax3 = ax.twiny()
    ax2.set_xlim(-4, 4)
    ax2.set_ylim(-4, 4)
    ax3.set_xlim(-4, 4)
    ax3.set_ylim(-4, 4)
    ax.set_xlabel("Bottom - Phase 2 (Indian Ocean) Phase 3", fontsize=20, labelpad=20)
    ax.set_ylabel("Left - Phase 1 (Western Hemisphere, Africa) Phase 8", fontsize=20, labelpad=20)
    ax2.set_ylabel("Right - Phase 5 (Maritime) Phase 4", fontsize=20, labelpad=40, rotation=270)
    ax3.set_xlabel("Top - Phase 7 (Western Pacific) Phase 6", fontsize=20, labelpad=20)
    
    # add amplitude circle
    amplitude_circle = plt.Circle((0, 0), 1, color=color_dm, fill=False)
    ax.add_patch(amplitude_circle)
    
    # add phase diagram lines
    ## Phase 1-8
    phase_line = lines.Line2D([-4, -1],
                               [0, 0],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))
    ax.add_line(phase_line)
    ## Phase 1-2
    phase_line = lines.Line2D([np.cos(3 * np.pi / 4), -4],
                               [np.sin(7 * np.pi / 4), -4],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))
    ax.add_line(phase_line)
    ## Phase 2-3
    phase_line = lines.Line2D([0, 0],
                               [-1, -4],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))

    ax.add_line(phase_line)
    ## Phase 3-4
    phase_line = lines.Line2D([np.cos(np.pi / 4), 4],
                               [np.sin(7 * np.pi / 4), -4],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))
    ax.add_line(phase_line)
    ## Phase 4-5
    phase_line = lines.Line2D([1, 4],
                               [0, 0],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))

    ax.add_line(phase_line)
    ## Phase 5-6
    phase_line = lines.Line2D([np.cos(np.pi / 4), 4],
                               [np.sin(np.pi / 4), 4],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))
    ax.add_line(phase_line)
    ## Phase 6-7
    phase_line = lines.Line2D([0, 0],
                               [1, 4],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))
    ax.add_line(phase_line)
    ## Phase 7-8
    phase_line = lines.Line2D([np.cos(3 * np.pi / 4), -4],
                               [np.sin(np.pi / 4), 4],
                               c=color_dm,
                               linestyle="--",
                               dashes=(5, 5))
    ax.add_line(phase_line)
    
    # add tick marks
    full_tickmarks = np.arange(-4, 4.5, 0.5)
    ax.set_xticks(full_tickmarks)
    ax.set_yticks(full_tickmarks)
    # hide tickmarks for top/right axis)
    ax2.set_yticks(full_tickmarks)
    ax3.set_xticks(full_tickmarks)
    
    # add (optional) phase number text in each segment
    ax.text(-3, -1.5, "1", color=color_dm, fontsize=20)
    ax.text(-1.5, -3, "2", color=color_dm, fontsize=20)
    ax.text(1.5, -3, "3", color=color_dm, fontsize=20)
    ax.text(3, -1.5, "4", color=color_dm, fontsize=20)
    ax.text(3, 1.5, "5", color=color_dm, fontsize=20)
    ax.text(1.5, 3, "6", color=color_dm, fontsize=20)
    ax.text(-1.5, 3, "7", color=color_dm, fontsize=20)
    ax.text(-3, 1.5, "8", color=color_dm, fontsize=20)
    
    
    # plot with title
    plt.title(f"MJO Phase: 15S-15N: {start_date}-{end_date}", fontsize=30, fontweight="bold", pad=20)
    
    plt.show()

In [ ]:
mjo_basemap(is_darkmode=True)

# MJO 4: Create MJO Lag Plots

Via `mjo_lag_lat_lon.ncl`

In [ ]:
#var_types = ["pr", "u850"]
var_types = ["prect", "u850"]
#file_dir = WORK_DIR + "/model/"
file_dir = WORK_DIR + DATADIR + "anomaly/" + CASENAME
filename_pr = file_dir  + ".prect.day.anom.nc"
filename_u850 = file_dir  + ".u850.day.anom.nc"

nameSeason = ["winter", "summer", "annual"]

In [ ]:
# Indian Ocean base region
nameRegion = "IO"
latS_IO = -10
latN_IO = 5
lonL_IO = 75
lonR_IO = 100

# global subset
latS_globe = -30
latN_globe = 30

# latitude band (lag, lon)
latn = 10
lats = -10

# longitude band for (lag, lat)
lonl = 80
lonr = 100

# output
pltName = CASENAME + ".MJO.lag.lat.lon"
pltType = "ps" # alteratively, x11, ps, eps, pdf, png
pltDir = file_dir + "/PS/" # output plot directory

### Fixed Lanczos Filter Weights

Create bandpass filter from weights

See: [Lanczos Filter Weights](https://www.ncl.ucar.edu/Applications/mjoclivar.shtml)

```
When needed, the weights for the suggested 20-100 day bandpass Lanczos filter are generated 'on-the-fly' using:

  ihp      = 2                             ; bpf=>band pass filter
  nWgt     = 201
  sigma    = 1.0                           ; Lanczos sigma
  fca      = 1./100.
  fcb      = 1./20.
  wgt      = filwgts_lanczos (nWgt, ihp, fca, fcb, sigma )

```

See `generate_lanczos_filter_weights.ncl` which generates `lanczos_filter_weights_output.txt`

In [ ]:
weights = lanczos_filter_weights()
weights

### Preciptation
- Time indices cooresponding to the desired time window
- Reader user specified period

In [ ]:
file_pr = xr.open_dataset(filename_pr)
file_pr

In [ ]:
pr = file_pr[var_types[0].upper()].sel(lat=slice(latS_globe, latN_globe)) # L89 (mjo_lag_lat_lon.ncl)
pr

In [ ]:
time_pr = pr["time"]
date_pr = pr['time'].values.tolist()
time_pr

In [ ]:
wypr = pr["lat"].sel(lat=slice(latS_IO, latN_IO)) # L97 (mjo_lag_lat_lon.ncl)
wypr

In [ ]:
wypr = np.cos(0.017459 * wypr)
wypr

In [ ]:
twStrt = time_pr["time"].min().item()
twLast = time_pr["time"].max().item()
print(f"Time range from {twStrt} to {twLast}")

### U850 Anomalies
- Time indices cooresponding to the desired time window
- Reader user specified period

In [ ]:
file_u850 = xr.open_dataset(filename_u850)
file_u850

In [ ]:
u850 = file_u850[var_types[1].upper()].sel(lat=slice(latS_globe, latN_globe)) # L89 (mjo_lag_lat_lon.ncl)
u850

In [ ]:
time_u850 = u850["time"]
date_u850 = u850['time'].values.tolist()
time_u850

In [ ]:
wyu850 = u850["lat"].sel(lat=slice(latS_IO, latN_IO)) # L131 (mjo_lag_lat_lon.ncl)
wyu850

In [ ]:
wyu850 = np.cos(0.017459 * wyu850)
wyu850

In [ ]:
# ensure dates agree
for i in range(len(date_pr)):
    try:
        date_pr[i] == date_u850[i]
    except Exception:
        print("Date mismatch: exit")    

In [ ]:
# Create weighted area average of the base IO precipation series (time)
pr_sub = pr.sel(lat=slice(latS_IO, latN_IO), lon=slice(lonL_IO, lonR_IO)) # L150 (mjo_lag_lat_lon.ncl)
pr_weights = wypr.sel(lat=slice(latS_IO, latN_IO))
pr_weighted = pr_sub.weighted(pr_weights) # weighted average
PIO = pr_weighted.mean(dim=["lat", "lon"])
# remove overall trendline (dtrend)
poly_fit_pr = pr_sub.polyfit(dim="time", deg=1)
fit_pr = xr.polyval(pr_sub["time"], poly_fit_pr.polyfit_coefficients)
PIO = pr_sub - fit_pr
# apply filter for leftdim (wgt_runave_leftdim) 
#TODO: L152

# MJO 5: Calculate MJO Spectra

Via `mjo_spectra.ncl`

In [ ]:
var_types = ["pr", "rlut", "u200", "u850", "v200", "v850"]

file_dir = WORK_DIR + DATADIR + "anomaly/" + CASENAME
filename_pr = file_dir  + ".prect.day.anom.nc"
filename_rlut = file_dir + ".rlut.day.anom.nc"
filename_u200 = file_dir + ".u200.day.anom.nc"
filename_u850 = file_dir + ".u850.day.anom.nc"
filename_v200 = file_dir + ".v200.day.anom.nc"
filename_v850 = file_dir + ".v850.day.anom.nc"

file_list = [filename_pr, filename_rlut, filename_u200, filename_u850, filename_v200, filename_v850]

In [ ]:
nameSeason = ["winter", "summer"]

latS = -10
latN = 10
vName = "U_anom" # name of variable on the file

pltDir = WORK_DIR + "/model/PS/"
pltType = "ps"

In [ ]:
# TODO

# MJO 6: Calculate Principle Component from EOF Analysis

Via `mjo_EOF_cal.ncl`

In [ ]:
# Combined EOFs

file_dir = WORK_DIR + "/model/"
pltDir = file_dir + "/PS/" # plot directory
pltType = "ps"
pltName = "mjoclivar"


neof = 2
latS = -20
latN = 20

anom_dir = WORK_DIR + "/data/" + CASENAME + "/day/anomaly/"
fileolr = anom_dir + CASENAME + ".flut.day.anom.nc"
fileu850 = anom_dir + CASENAME + ".u850.day.anom.nc"
fileu200 = anom_dir + CASENAME + ".u200.day.anom.nc"

### Fixed Lanczos Filter Weights

Create bandpass filter from weights

See: [Lanczos Filter Weights](https://www.ncl.ucar.edu/Applications/mjoclivar.shtml)

```
When needed, the weights for the suggested 20-100 day bandpass Lanczos filter are generated 'on-the-fly' using:

  ihp      = 2                             ; bpf=>band pass filter
  nWgt     = 201
  sigma    = 1.0                           ; Lanczos sigma
  fca      = 1./100.
  fcb      = 1./20.
  wgt      = filwgts_lanczos (nWgt, ihp, fca, fcb, sigma )

```

See `generate_lanczos_filter_weights.ncl` which generates `lanczos_filter_weights_output.txt`

In [ ]:
weights = lanczos_filter_weights()
weights

In [ ]:
file_olr = xr.open_dataset(fileolr)
file_olr

In [ ]:
# get indicies corresponding to the start/end times
TIME = file_olr["time"]
ymdStrt = file_olr["time"][file_olr.time.argmin()]
ymdLast = file_olr["time"][file_olr.time.argmax()]
yrStrt = ymdStrt["time"].item().year
yrLast = ymdLast["time"].item().year

In [ ]:
# read anomalies

# RLUT
print(fileolr)
work = file_olr["FLUT"].sel(lat=slice(latS, latN))
rightmost_dim = list(work.sizes.keys())[-1]
#print(f"rightmost dimension: {rightmost_dim}\n")
OLR = work.mean(dim=rightmost_dim) # dim_avg_wrap

In [ ]:
# read anomalies

# U850
print(fileu850)
file_u850 = xr.open_dataset(fileu850)
work = file_u850["U850"].sel(lat=slice(latS, latN))
rightmost_dim = list(work.sizes.keys())[-1]
#print(f"rightmost dimension: {rightmost_dim}\n")
U850 = work.mean(dim=rightmost_dim) # dim_avg_wrap

In [ ]:
# read anomalies

# U200
print(fileu200)
file_u200 = xr.open_dataset(fileu200)
work = file_u200["U200"].sel(lat=slice(latS, latN))
rightmost_dim = list(work.sizes.keys())[-1]
#print(f"rightmost dimension: {rightmost_dim}\n")
U200 = work.mean(dim=rightmost_dim) # dim_avg_wrap

In [ ]:
dimw = work.shape
print(dimw)
ntim = dimw[0]
nlat = dimw[1]
mlon = dimw[2]

In [ ]:
lon = file_u200["lon"]
time = file_u200["time"]

#TODO: L88

In [ ]:
# Apply the band pass filter to the original anomalies

# wgt_runave_wrap: calculates a weighted running average on the rightmost dimension and retains metadata
def wgt_runave_wrap(dataset):
    print("Calculating weighted running average...")
    rightmost_dim = list(dataset.sizes.keys())[-1]
    weighted_running_average = dataset.rolling(time=len(weights), center=True).mean().dropna("time")
    print(weighted_running_average)
    return weighted_running_average

rlut = wgt_runave_wrap(OLR)
u850 = wgt_runave_wrap(U850)
u200 = wgt_runave_wrap(U200)

In [ ]:
# TODO: remove means of band pass series (not necessary)
# L100-102

In [ ]:
# Compute the temporal variance

# dim_variance_Wrap -> Computes unbiased estimates of the variance of a variable's rightmost dimension
def dim_variance_wrap(dataset):
    print("Computing variance...")
    rightmost_dim = list(dataset.sizes.keys())[-1]
    variance = dataset.var(dim=rightmost_dim)
    print(variance)
    return variance
    
var_rlut = dim_variance_wrap(rlut)
var_u850 = dim_variance_wrap(u850)
var_u200 = dim_variance_wrap(u200)

In [ ]:
# Compute the zonal mean of the temporal variance

def dim_avg_wrap(dataset):
    print("Computing average...")
    rightmost_dim = list(dataset.sizes.keys())[-1]
    avg = dataset.mean(dim=rightmost_dim)
    print(avg)
    return avg

zavg_var_rlut = dim_avg_wrap(var_rlut)
zavg_var_u850 = dim_avg_wrap(var_u850)
zavg_var_u200 = dim_avg_wrap(var_u200)

In [ ]:
# Noramlize by sqrt(avg_var)

rlut = rlut/np.sqrt(zavg_var_rlut)
u850 = u850/np.sqrt(zavg_var_u850)
u200 = u200/np.sqrt(zavg_var_u200)

print(rlut)
print(u850)
print(u200)

In [ ]:
# Combine the noramlized data into variable
cdata = xr.DataArray(np.full((3*mlon, ntim), np.nan, dtype=np.float32))
print(cdata.shape)
print(cdata)

for m1 in range(mlon-1):
    print(cdata[m1,:])
    print(rlut[m1,:])
    print("\n")
    print(cdata.values[m1,:].shape)
    print(rlut[m1,:].shape)
    cdata.values[m1,:] = rlut.values[m1,:]
    #cdata[m1+mlon,:] = u850[m1,:]
    #cdata[m1+2*mlon,:] = u200[m1,:]


# MJO 7: MJO Life Cycle Composite

Via `mjo_life_cycle.ncl`